# Hakam — training v3

Exp 3 underfit: the new classifier head learned at the backbone's 1e-5. This round fixes that, then tests two ideas from prior work (see `docs/RELATED_WORK.md`).

1. **Runtime → Change runtime type → L4 GPU**
2. **Runtime → Run all** (about 80 minutes)

## 1. GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU - change the runtime type."
print(torch.cuda.get_device_name(0))

## 2. Code from GitHub, data and labels from Drive

In [ ]:
import shutil, zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
pkg = Path("/content/drive/MyDrive/hakam_colab")
work = Path("/content/hakam")
if work.exists():
    !cd /content/hakam && git pull -q
else:
    !git clone -q https://github.com/FerasMad/hakam.git /content/hakam

for split in ["Train", "Valid", "Test"]:
    if (work / "data" / "mvfouls" / split).exists():
        continue
    local = Path("/content") / f"{split}.zip"
    shutil.copy(pkg / "data" / f"{split}.zip", local)
    zipfile.ZipFile(local).extractall(work / "data" / "mvfouls")
    local.unlink()
    print(split, "ready")

labels = work / "artifacts" / "preprocessing" / "private"
labels.mkdir(parents=True, exist_ok=True)
for f in (pkg / "manifests").glob("*.csv"):
    shutil.copy(f, labels)
print("labels ready")

## 3. Install and cache frames

In [ ]:
%cd /content/hakam
!pip install -q transformers
!python scripts/cache_frames.py --splits train valid test

## 4. Experiment 5 — separate learning rate for the head
Same as Exp 3, but the head learns at 1e-3 with one epoch of warmup, 8 epochs.

In [ ]:
!python scripts/round3.py exp5

## 5. Experiment 6 — frames packed around the foul
16 frames over 1.3 s centred on the foul (stride 2) instead of 2.56 s (stride 4). VARS used 1 s; CAS-FD gained 12 points from contact-centred sampling.

In [ ]:
!python scripts/round3.py exp6

## 6. Experiment 7 — bigger backbone
VideoMAE-base on whichever window won.

In [ ]:
!python scripts/round3.py exp7

## 7. Experiment 8 — offence stage
Best settings so far, on the first question the contract answers.

In [ ]:
!python scripts/round3.py exp8

## 8. Results and save to Drive
Compare runs by `last3_mean`. A gain needs about 6 points, or intervals that don't overlap.

In [ ]:
!python scripts/summarise_runs.py --save /content/drive/MyDrive/hakam_colab/runs